In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments
from datasets import Dataset
import torch
import json
import os

# Load dataset (from JSON converted from Excel)
with open("training_data.json") as f:
    data = json.load(f)

dataset = Dataset.from_list(data)

model_name = "distilgpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

def tokenize(example):
    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

dataset = dataset.map(tokenize)

training_args = TrainingArguments(
    output_dir="./campushub_model",
    per_device_train_batch_size=4,
    num_train_epochs=3,
    logging_dir="./logs",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset
)

trainer.train()

model.save_pretrained("./campushub_model")
tokenizer.save_pretrained("./campushub_model")
print("Current working directory:", os.getcwd())
